# NLDAS3 Agroclimate Climatology (Polygon, 2016-2025)

This notebook:
- Reads monthly NLDAS3 indicator files from S3
- Filters to 2016-2025
- Clips to a polygon (same region logic as your GRIDMET workflow)
- Computes monthly climatology mean and standard deviation
- Exports one GeoTIFF per month with derived bands

In [ ]:
# Uncomment if needed
# %pip install xarray s3fs h5netcdf rioxarray geopandas shapely matplotlib pandas

In [8]:
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import s3fs
import xarray as xr
from shapely.geometry import box, mapping, shape

import rioxarray  # noqa: F401 (register rio accessor)

In [9]:
import dask
import distributed
print("dask", dask.__version__)
print("distributed", distributed.__version__)

dask 2024.8.0
distributed 2024.8.0


In [10]:
import h5py
print("h5py", h5py.__version__)

h5py 3.14.0


In [11]:
# ------------------------------
# Parameters
# ------------------------------
bucket_path = "water-insight-public-test/gdd_frost_heat_indicators/"
start_date = "2016-01-01"
end_date = "2025-12-31"

# Use the same Alabama buffered rectangle from your GRIDMET script by default
default_bbox = (-88.5, 30.1, -84.8, 35.1)

# Option A: Provide polygon GeoJSON path
polygon_geojson_path = None  # example: "./polygon.geojson"

# Option B: Provide WKT string
polygon_wkt = None

out_dir = Path("./nldas3-data/climatology")
out_dir.mkdir(parents=True, exist_ok=True)

variables = ["gdd", "hsd", "frost_days"]

In [5]:
# ------------------------------
# Set AWS credentials in THIS notebook kernel (only if not already set)
# ------------------------------
# This avoids relying on shell exports that the kernel may not inherit.
import os
import getpass

if not os.getenv("AWS_ACCESS_KEY_ID"):
    os.environ["AWS_ACCESS_KEY_ID"] = input("Enter AWS_ACCESS_KEY_ID: ").strip()

if not os.getenv("AWS_SECRET_ACCESS_KEY"):
    os.environ["AWS_SECRET_ACCESS_KEY"] = getpass.getpass("Enter AWS_SECRET_ACCESS_KEY: ")

# STS temporary keys (ASIA...) require a session token.
if os.getenv("AWS_ACCESS_KEY_ID", "").startswith("ASIA") and not os.getenv("AWS_SESSION_TOKEN"):
    os.environ["AWS_SESSION_TOKEN"] = getpass.getpass("Enter AWS_SESSION_TOKEN: ")

# Optional: profile for shared credentials file usage
if not os.getenv("AWS_PROFILE"):
    _profile = input("Optional AWS_PROFILE (press Enter to skip): ").strip()
    if _profile:
        os.environ["AWS_PROFILE"] = _profile

masked_key = os.getenv("AWS_ACCESS_KEY_ID", "")
if len(masked_key) > 6:
    masked_key = masked_key[:4] + "..." + masked_key[-2:]
print("AWS credentials set in kernel. Key:", masked_key)

AWS credentials set in kernel. Key: ASIA...ZA


In [13]:
# ------------------------------
# Read file list from S3 and open dataset
# ------------------------------
# Use authenticated access when credentials are available in the kernel environment.
# If your creds were exported in a terminal after kernel start, restart the kernel first.
import os
import re
import s3fs
import xarray as xr

aws_key = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret = os.getenv("AWS_SECRET_ACCESS_KEY")
aws_token = os.getenv("AWS_SESSION_TOKEN")
aws_profile = os.getenv("AWS_PROFILE")

storage_options = {}
if aws_profile:
    storage_options["profile"] = aws_profile
if aws_key and aws_secret:
    storage_options.update({
        "key": aws_key,
        "secret": aws_secret,
    })
if aws_token:
    storage_options["token"] = aws_token

try:
    fs = s3fs.S3FileSystem(anon=False, **storage_options)
    files = sorted(fs.glob(bucket_path + "*.nc"))
except PermissionError as exc:
    raise PermissionError(
        "Access Denied while listing S3 files. The notebook kernel likely does not have valid AWS credentials "
        "or bucket list permissions. Restart kernel after exporting AWS_* vars, or set them inside "
        "the notebook with %env before this cell."
    ) from exc

print(f"Found {len(files)} files in bucket")
print(files[:5])

if len(files) == 0:
    raise RuntimeError("No files found in bucket_path. Check permissions/path.")

# Filter file list by YYYYMM in filename, e.g. agroclim_indicator-202107.nc
start_ym = start_date[:7].replace("-", "")
end_ym = end_date[:7].replace("-", "")

def _extract_ym(path):
    m = re.search(r"(\d{6})\.nc$", path)
    return m.group(1) if m else None

selected_files = [
    p for p in files
    if (_extract_ym(p) is not None and start_ym <= _extract_ym(p) <= end_ym)
]

print(f"Selected {len(selected_files)} files for {start_date} to {end_date}")
print(selected_files[:5])

if len(selected_files) == 0:
    raise RuntimeError("No monthly files matched the requested date range.")

s3_urls = ["s3://" + p for p in selected_files]

try:
    # Fast path: let xarray/fsspec handle s3 URLs with explicit storage options
    ds = xr.open_mfdataset(
        s3_urls,
        engine="h5netcdf",
        combine="by_coords",
        parallel=True,
        chunks="auto",
        backend_kwargs={"storage_options": storage_options},
    )
except Exception:
    # Fallback path for environments where direct s3:// with h5netcdf is unsupported
    open_files = [fs.open(p, mode="rb") for p in selected_files]
    ds = xr.open_mfdataset(
        open_files,
        engine="h5netcdf",
        combine="by_coords",
        parallel=True,
        chunks="auto",
    )

print(ds)
print("Data variables:", list(ds.data_vars))

Found 276 files
['water-insight-public-test/gdd_frost_heat_indicators/agroclim_indicator-200101.nc', 'water-insight-public-test/gdd_frost_heat_indicators/agroclim_indicator-200102.nc', 'water-insight-public-test/gdd_frost_heat_indicators/agroclim_indicator-200103.nc', 'water-insight-public-test/gdd_frost_heat_indicators/agroclim_indicator-200104.nc', 'water-insight-public-test/gdd_frost_heat_indicators/agroclim_indicator-200105.nc']


KeyboardInterrupt: 

In [ ]:
# ------------------------------
# Time filter (2016-2025)
# ------------------------------
ds = ds.sel(time=slice(start_date, end_date))
print("Filtered time range:", str(ds.time.min().values), "to", str(ds.time.max().values))

missing_vars = [v for v in variables if v not in ds.data_vars]
if missing_vars:
    raise ValueError(f"Missing expected variables: {missing_vars}")

ds = ds[variables]

In [ ]:
# ------------------------------
# Build polygon and clip
# ------------------------------
if polygon_geojson_path:
    polygon_gdf = gpd.read_file(polygon_geojson_path).to_crs("EPSG:4326")
elif polygon_wkt:
    polygon_gdf = gpd.GeoDataFrame(geometry=[shape(mapping(box(*default_bbox)))], crs="EPSG:4326")
    polygon_gdf.loc[0, "geometry"] = gpd.GeoSeries.from_wkt([polygon_wkt], crs="EPSG:4326").iloc[0]
else:
    polygon_gdf = gpd.GeoDataFrame(geometry=[box(*default_bbox)], crs="EPSG:4326")

# xarray/rioxarray spatial metadata
ds = ds.rio.set_spatial_dims(x_dim="lon", y_dim="lat", inplace=False).rio.write_crs("EPSG:4326", inplace=False)
ds_clip = ds.rio.clip(polygon_gdf.geometry.apply(mapping), polygon_gdf.crs, drop=True)

print(ds_clip)

In [ ]:
# ------------------------------
# Monthly climatology (mean + std)
# ------------------------------
monthly_mean = ds_clip.groupby("time.month").mean("time", skipna=True)
monthly_std = ds_clip.groupby("time.month").std("time", skipna=True)

monthly_mean

In [ ]:
# ------------------------------
# Export one monthly multi-band GeoTIFF
# Bands: gdd_mean, hsd_mean, frost_days_mean, gdd_std, hsd_std, frost_days_std
# ------------------------------
band_order = [
    "gdd_mean", "hsd_mean", "frost_days_mean",
    "gdd_std", "hsd_std", "frost_days_std",
]

for m in range(1, 13):
    ds_m = xr.Dataset({
        "gdd_mean": monthly_mean["gdd"].sel(month=m),
        "hsd_mean": monthly_mean["hsd"].sel(month=m),
        "frost_days_mean": monthly_mean["frost_days"].sel(month=m),
        "gdd_std": monthly_std["gdd"].sel(month=m),
        "hsd_std": monthly_std["hsd"].sel(month=m),
        "frost_days_std": monthly_std["frost_days"].sel(month=m),
    })

    da = ds_m[band_order].to_array(dim="band")
    da = da.rename({"lat": "y", "lon": "x"})

    # Ensure north-up GeoTIFF (negative y resolution); required by some zonal tools.
    da = da.sortby("y", ascending=False)

    da = da.rio.write_crs("EPSG:4326", inplace=False)

    out_path = out_dir / f"NLDAS3_Climatology_Month_{m:02d}.tif"
    da.rio.to_raster(out_path)

    print("Wrote", out_path)

# Save band metadata mapping
band_meta = {"bands": {str(i + 1): name for i, name in enumerate(band_order)}}
with open(out_dir / "NLDAS3_Climatology_Bands.json", "w", encoding="utf-8") as f:
    json.dump(band_meta, f, indent=2)

print("Done exporting monthly climatology GeoTIFFs")

In [ ]:
# ------------------------------
# Quick map: first 12 monthly GDD means
# ------------------------------
fig, axes = plt.subplots(3, 4, figsize=(16, 10), constrained_layout=True)
axes = axes.flatten()

for i, m in enumerate(range(1, 13)):
    monthly_mean["gdd"].sel(month=m).plot(ax=axes[i], cmap="turbo", add_colorbar=False)
    axes[i].set_title(f"GDD Mean - Month {m:02d}")

cbar = fig.colorbar(axes[0].collections[0], ax=axes, orientation="horizontal", fraction=0.05, pad=0.06)
cbar.set_label("GDD")
plt.show()

In [ ]:
# ------------------------------
# Monthly cycle over polygon (spatial mean)
# ------------------------------
cycle_df = pd.DataFrame({
    "month": list(range(1, 13)),
    "gdd_mean": [float(monthly_mean["gdd"].sel(month=m).mean().values) for m in range(1, 13)],
    "hsd_mean": [float(monthly_mean["hsd"].sel(month=m).mean().values) for m in range(1, 13)],
    "frost_days_mean": [float(monthly_mean["frost_days"].sel(month=m).mean().values) for m in range(1, 13)],
})

ax = cycle_df.plot(x="month", y=["gdd_mean", "hsd_mean", "frost_days_mean"], marker="o", figsize=(10, 5))
ax.set_title("NLDAS3 Monthly Climatology Cycle (Polygon Mean)")
ax.set_xlabel("Month")
ax.set_ylabel("Indicator value")
ax.grid(True, alpha=0.3)
plt.show()

cycle_df